<style>
h1, h1 *, 
h2, h2 *, 
h3, h3 *, 
h4, h4 *, 
h5, h5 *, 
h6, h6 * {
  font-family: "Times New Roman", Times, serif !important;

</style>

### **Book Recommendation System**

---

In [1]:
import sys
!{sys.executable} -m pip install requests

In [2]:
import sys
!{sys.executable} -m pip install nltk

In [3]:
import sys
!{sys.executable} -m pip install Pillow

In [4]:
import os
import re
import nltk
import requests
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from nltk.corpus import stopwords
nltk.download('stopwords')

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from PIL import Image
warnings.filterwarnings('ignore')
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dilan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
books = pd.read_csv('Book Recommendation System - Dataset (Preprocessed).csv')
books.head()

,Unnamed: 0,user_id,location,age,isbn,rating,book_title,book_author,year_of_publication,publisher,img_s,img_m,img_l,Summary,Language,Category,city,state,country
0,0,2,"stockton, california, usa",18.0000,0195153448,0,Classical Mythology,Mark P. O. Morford,2002.0,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,Provides an introduction to classical myths pl...,en,['Social Science'],stockton,california,usa
1,1,8,"timmins, ontario, canada",34.7439,0002005018,5,Clara Callan,Richard Bruce Wright,2001.0,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,"In a small town in Canada, Clara Callan reluct...",en,['Actresses'],timmins,ontario,canada
2,2,11400,"ottawa, ontario, canada",49.0000,0002005018,0,Clara Callan,Richard Bruce Wright,2001.0,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,"In a small town in Canada, Clara Callan reluct...",en,['Actresses'],ottawa,ontario,canada
3,3,11676,"n/a, n/a, n/a",34.7439,0002005018,8,Clara Callan,Richard Bruce Wright,2001.0,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,"In a small town in Canada, Clara Callan reluct...",en,['Actresses'],NaN,NaN,NaN
4,4,41385,"sudbury, ontario, canada",34.7439,0002005018,0,Clara Callan,Richard Bruce Wright,2001.0,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,"In a small town in Canada, Clara Callan reluct...",en,['Actresses'],sudbury,ontario,canada


In [6]:
df = books.copy()

In [7]:
df.dropna(inplace = True)

In [8]:
df.reset_index(drop = True, inplace = True)

In [9]:
df.drop(columns = ['Unnamed: 0', 'location', 'isbn', 'img_s', 'img_m',
                   'city', 'age', 'state', 'Language', 'country', 
                   'year_of_publication'], axis = 1, inplace = True)

In [10]:
df['Category'].value_counts()

Category
9                                386063
['Fiction']                      367545
['Juvenile Fiction']              37884
['Biography & Autobiography']     21306
['Humor']                          8135
                                  ...  
['Bloodsucking animals']              1
['Women composers']                   1
['Utah']                              1
['Hospitals']                         1
['Database searching']                1
Name: count, Length: 6347, dtype: int64

In [11]:
df.drop(index = df[df['Category'] == '9'].index, inplace = True)

In [12]:
df.drop(index = df[df['rating'] == 0].index, inplace = True)

In [13]:
df['Category'] = df['Category'].apply(lambda x: re.sub('[\W_]+', ' ', x).strip())

In [14]:
df.head()

,user_id,rating,book_title,book_author,publisher,img_l,Summary,Category
1,8,5,Clara Callan,Richard Bruce Wright,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,"In a small town in Canada, Clara Callan reluct...",Actresses
4,67544,8,Clara Callan,Richard Bruce Wright,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,"In a small town in Canada, Clara Callan reluct...",Actresses
7,123629,9,Clara Callan,Richard Bruce Wright,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,"In a small town in Canada, Clara Callan reluct...",Actresses
9,200273,8,Clara Callan,Richard Bruce Wright,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,"In a small town in Canada, Clara Callan reluct...",Actresses
10,210926,9,Clara Callan,Richard Bruce Wright,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,"In a small town in Canada, Clara Callan reluct...",Actresses


In [15]:
plt.rcParams["font.family"] = "Times New Roman"

<style>
h1, h1 *, 
h2, h2 *, 
h3, h3 *, 
h4, h4 *, 
h5, h5 *, 
h6, h6 * {
  font-family: "Times New Roman", Times, serif !important;
}
</style>

#### **Item-Based Collaborative Filtering**

In [16]:
def item_based_recommender(book_title):

    book_title = str(book_title)
    
    if book_title in df['book_title'].values:

        rating_counts = df['book_title'].value_counts()
        rare_books = rating_counts[rating_counts <= 180].index
        common_books = df[~ df['book_title'].isin(rare_books)]

        if book_title in rare_books:

            random = pd.Series(common_books['book_title'].unique()).sample(2).values
            print('There are no recommendations for this book')
            print('Try: \n')
            print('{}'.format(random[0]), '\n')
            print('{}'.format(random[1]), '\n')

        else:

            user_book_df = common_books.pivot_table(index = ['user_id'],
                                                    columns = ['book_title'],
                                                    values = 'rating')

            book = user_book_df[book_title]
            recommendation_data = (user_book_df.corrwith(book).sort_values(ascending = False).reset_index())
            recommendation_data.columns = ['book_title', 'corr']

            if book_title in [book for book in recommendation_data['book_title']]:
                recommendation_data = recommendation_data.drop(recommendation_data[recommendation_data['book_title'] == book_title].index[0])

            low_rating = []
            for i in recommendation_data['book_title']:
                if df[df['book_title'] == i]['rating'].mean() < 5:
                    low_rating.append(i)
            if recommendation_data.shape[0] - len(low_rating) > 5:
                recommendation_data = recommendation_data[~ recommendation_data['book_title'].isin(low_rating)]

            recommendation_data = recommendation_data.head(5)
            recommendation_data['rating'] = recommendation_data['book_title'].apply(lambda x: round(df[df['book_title'] == x]['rating'].mean(), 1))
            recommendation_data = recommendation_data[['book_title', 'rating', 'corr']]
            recommendation_data.columns = ['Recommended Book', 'Average Rating', 'Similarity Score']

            print(f'Recommendations for {book_title}: \n')

            display(recommendation_data)

    else:

        print('Unable to find book in the dataset. Please try again.')

In [17]:
item_based_recommender('The Street Lawyer')

Recommendations for The Street Lawyer: 



,Recommended Book,Average Rating,Similarity Score
1,Girl with a Pearl Earring,8.0,0.960769
2,"Tuesdays with Morrie: An Old Man, a Young Man,...",8.6,0.845154
3,The Testament,7.5,0.844485
4,The Red Tent (Bestselling Backlist),8.2,0.817424
5,The Secret Life of Bees,8.5,0.774597


In [18]:
item_based_recommender('Fahrenheit 451')

Recommendations for Fahrenheit 451: 



,Recommended Book,Average Rating,Similarity Score
1,The Five People You Meet in Heaven,8.0,0.944911
2,Interview with the Vampire,7.8,0.939336
3,The Da Vinci Code,8.4,0.867722
4,Bridget Jones's Diary,7.6,0.835619
5,Harry Potter and the Sorcerer's Stone (Harry P...,8.9,0.761885


In [19]:
item_based_recommender('Divine Secrets of the Ya-Ya Sisterhood: A Novel')

Recommendations for Divine Secrets of the Ya-Ya Sisterhood: A Novel: 



,Recommended Book,Average Rating,Similarity Score
1,The Fellowship of the Ring (The Lord of the Ri...,8.9,0.919145
2,Snow Falling on Cedars,7.8,0.754255
3,A Time to Kill,8.0,0.659919
4,The Nanny Diaries: A Novel,7.5,0.608625
5,The Testament,7.5,0.500000


<style>
h1, h1 *, 
h2, h2 *, 
h3, h3 *, 
h4, h4 *, 
h5, h5 *, 
h6, h6 * {
  font-family: "Times New Roman", Times, serif !important;
}
</style>

#### **Content-Based Collaborative Filtering**

In [20]:
def content_based_recommender(book_title):

    book_title = str(book_title)

    if book_title in df['book_title'].values:

        rating_counts = df['book_title'].value_counts()
        rare_books = rating_counts[rating_counts <= 100].index
        common_books = df[~ df['book_title'].isin(rare_books)]

        if book_title in rare_books:

            random_books = pd.Series(common_books['book_title'].unique()).sample(2).values
            print('There are no recommendations for this book')
            print('Try: \n')
            print(random_books[0])
            print(random_books[1])

        else:

            common_books = common_books.drop_duplicates(subset = ['book_title'])
            common_books.reset_index(drop = True, inplace = True)
            common_books['index'] = range(common_books.shape[0])
            target_columns = ['book_title', 'book_author', 'publisher', 'Category']
            common_books['combined_features'] = (common_books[target_columns].fillna('').astype(str).agg(' '.join, axis = 1))

            cv = CountVectorizer()
            count_matrix = cv.fit_transform(common_books['combined_features'])
            cos_sim = cosine_similarity(count_matrix)
            index = common_books[common_books['book_title'] == book_title]['index'].values[0]
            similarity_books = list(enumerate(cos_sim[index]))
            sorted_similarity_books = sorted(similarity_books, key = lambda x: x[1], reverse = True)[1:6]

            recommendations = []

            for book_index, similarity in sorted_similarity_books:
                
                recommend_title = common_books[common_books['index'] == book_index]['book_title'].values[0]
                recommendations.append({'Recommended Book': recommend_title,
                                        'Average Rating': round(df[df['book_title'] == recommend_title]['rating'].mean(), 1),
                                        'Similarity Score': round(similarity, 3)})
                
            recommendation_table = pd.DataFrame(recommendations)
            print(f'Recommendations for {book_title}: \n')
            display(recommendation_table)

    else:
        
        print('Unable to find book in the dataset. Please try again.')            

In [21]:
content_based_recommender('Animal Farm')

Recommendations for Animal Farm: 



,Recommended Book,Average Rating,Similarity Score
0,Neverwhere,8.0,0.183
1,Dreamcatcher,7.1,0.183
2,Outlander,8.6,0.183
3,The Testament,7.5,0.167
4,Airframe,7.5,0.167


In [22]:
content_based_recommender('1st to Die: A Novel')

Recommendations for 1st to Die: A Novel: 



,Recommended Book,Average Rating,Similarity Score
0,When the Wind Blows,7.5,0.556
1,Along Came a Spider (Alex Cross Novels),8.1,0.402
2,A Walk to Remember,7.9,0.354
3,Suzanne's Diary for Nicholas,7.6,0.333
4,The Bridges of Madison County,7.5,0.302


In [23]:
content_based_recommender('Harry Potter and the Order of the Phoenix (Book 5)')

Recommendations for Harry Potter and the Order of the Phoenix (Book 5): 



,Recommended Book,Average Rating,Similarity Score
0,Harry Potter and the Chamber of Secrets (Book 2),8.9,0.820
1,Harry Potter and the Prisoner of Azkaban (Book 3),9.1,0.820
2,Harry Potter and the Goblet of Fire (Book 4),9.2,0.820
3,Harry Potter and the Sorcerer's Stone (Book 1),9.1,0.778
4,Harry Potter and the Sorcerer's Stone (Harry P...,8.9,0.592


<style>
h1, h1 *, 
h2, h2 *, 
h3, h3 *, 
h4, h4 *, 
h5, h5 *, 
h6, h6 * {
  font-family: "Times New Roman", Times, serif !important;
}
</style>

#### **Custom Recommender**

In [24]:
import nltk

nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\dilan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dilan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [25]:
def custom_recommender(book_title):

    book_title = str(book_title)

    if book_title not in df['book_title'].values:
        print('Unable to find book in the dataset. Please try again.')
        return

    rating_counts = df['book_title'].value_counts()
    rare_books = rating_counts[rating_counts <= 180].index
    common_books = df[~ df['book_title'].isin(rare_books)].copy()

    if book_title in rare_books:
        random_books = (pd.Series(common_books['book_title'].unique()).sample(2).values)
        print('There are no recommendations for this book')
        print('Try: \n')
        print(random_books[0])
        print(random_books[1])
        return

    user_book_df = common_books.pivot_table(index = 'user_id', 
                                            columns = 'book_title', 
                                            values = 'rating')
    book_ratings = user_book_df[book_title]
    item_similarity = (user_book_df.corrwith(book_ratings).sort_values(ascending = False).reset_index())
    item_similarity.columns = ['book_title', 'corr']
    item_similarity = item_similarity[item_similarity['book_title'] != book_title]

    low_rating_books = []
    for book in item_similarity['book_title']:
        average_rating = df[df['book_title'] == book]['rating'].mean()
        if average_rating < 5:
            low_rating_books.append(book)
    if item_similarity.shape[0] - len(low_rating_books) > 5:
        item_similarity = item_similarity[~ item_similarity['book_title'].isin(low_rating_books)]

    item_recommendation = (item_similarity.head(1))
    recommended_books = []
    for book in item_recommendation['book_title']:
        recommended_books.append(book)

    df_new = df[~ df['book_title'].isin(recommended_books)].copy()
    rating_counts = df_new['book_title'].value_counts()
    rare_books = rating_counts[rating_counts <= 100].index
    common_books = df_new[~ df_new['book_title'].isin(rare_books)].copy()
    common_books = common_books.drop_duplicates(subset = ['book_title'])
    common_books.reset_index(drop = True, inplace = True)
    common_books['index'] = range(common_books.shape[0])
    target_columns = ['book_title', 'book_author', 'publisher', 'Category']
    common_books['combined_features'] = (common_books[target_columns].fillna('').astype(str).agg(' '.join, axis = 1))

    cv = CountVectorizer()
    count_matrix = cv.fit_transform(common_books['combined_features'])
    cos_sim = cosine_similarity(count_matrix)
    book_index = common_books[common_books['book_title'] == book_title]['index'].values[0]
    similarity_scores = list(enumerate(cos_sim[book_index]))
    similarity_scores = sorted(similarity_scores, key = lambda x: x[1], reverse = True)[1:2]

    for index, score in similarity_scores:
        recommended_books.append(common_books[common_books['index'] == index]['book_title'].values[0])

    df_new = df[~ df['book_title'].isin(recommended_books)].copy()
    rating_counts = df_new['book_title'].value_counts()
    rare_books = rating_counts[rating_counts <= 100].index
    common_books = df_new[~ df_new['book_title'].isin(rare_books)].copy()
    common_books = common_books.drop_duplicates(subset = ['book_title'])
    common_books.reset_index(drop = True, inplace = True)
    common_books['index'] = range(common_books.shape[0])

    cleaned_summary = []
    stop_words = set(stopwords.words('english'))
    for summary in common_books['Summary'].fillna(''):
        summary = re.sub('[^a-zA-Z]', ' ', str(summary)).lower()
        words = nltk.word_tokenize(summary)
        words = [word for word in words if word not in stop_words]
        cleaned_summary.append(' '.join(words))
    common_books['Summary'] = cleaned_summary
    cv = CountVectorizer()
    summary_matrix = cv.fit_transform(common_books['Summary'])
    summary_similarity = cosine_similarity(summary_matrix)
    book_index = common_books[common_books['book_title'] == book_title]['index'].values[0]
    similarity_scores = list(enumerate(summary_similarity[book_index]))
    similarity_scores = sorted(similarity_scores, key = lambda x: x[1], reverse = True)[1:3]
    
    for index, score in similarity_scores:
        recommended_books.append(common_books[common_books['index'] == index]['book_title'].values[0])

    category = common_books[common_books['book_title'] == book_title]['Category'].values[0]
    category_books = (common_books[common_books['Category'] == category].groupby('book_title')['rating'].mean().reset_index())
    category_books = category_books[category_books['book_title'] != book_title]

    if category_books.shape[0] > 0:
        top_category_book = (category_books.sort_values('rating', ascending = False).iloc[0]['book_title'])
        recommended_books.append(top_category_book)

    recommended_books = list(dict.fromkeys(recommended_books))
    recommendation_table = pd.DataFrame({'Recommended Book': recommended_books})
    recommendation_table['Average Rating'] = (recommendation_table['Recommended Book'].apply(lambda x: round(df[df['book_title'] == x]['rating'].mean(), 1)))
    recommendation_table['Similarity Score'] = ([round(1 - (i * 0.1), 3) for i in range(len(recommendation_table))])

    print(f'Recommendations for {book_title}: \n')
    display(recommendation_table)

In [26]:
custom_recommender('The Summons')

Recommendations for The Summons: 



,Recommended Book,Average Rating,Similarity Score
0,Harry Potter and the Order of the Phoenix (Boo...,9.0,1.0
1,The Chamber,7.4,0.9
2,Cold Mountain : A Novel,7.7,0.8
3,ANGELA'S ASHES,8.4,0.7
4,American Gods,8.0,0.6


In [27]:
custom_recommender('Snow Falling on Cedars')

Recommendations for Snow Falling on Cedars: 



,Recommended Book,Average Rating,Similarity Score
0,A Painted House,7.4,1.0
1,Empire Falls,7.6,0.9
2,Midnight in the Garden of Good and Evil: A Sav...,7.8,0.8
3,Angela's Ashes (MMP) : A Memoir,8.0,0.7
4,American Gods,8.0,0.6


In [28]:
custom_recommender("Tuesdays with Morrie: An Old Man, a Young Man, and Life's Greatest Lesson")

Recommendations for Tuesdays with Morrie: An Old Man, a Young Man, and Life's Greatest Lesson: 



,Recommended Book,Average Rating,Similarity Score
0,A Time to Kill,8.0,1.0
1,ANGELA'S ASHES,8.4,0.9
2,The Five People You Meet in Heaven,8.0,0.8
3,Interview with the Vampire,7.8,0.7
4,Angela's Ashes (MMP) : A Memoir,8.0,0.6
